In [ ]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
import re

df = pd.read_excel("CSV tarea MIA_Carmen.xlsx")
# Descargar las stopwords (solo se hace una vez)
nltk.download('stopwords')
stop_words = set(stopwords.words('english')) # Las reseñas están en inglés

def limpiar_texto(texto):
    # 1. Pasar a minúsculas
    texto = texto.lower()
    # 2. Eliminar puntuación y caracteres especiales
    texto = re.sub(r'[^\w\s]', '', texto)
    # 3. Eliminar stopwords
    palabras = texto.split()
    texto_limpio = [w for w in palabras if w not in stop_words]
    return " ".join(texto_limpio)

# Aplicar a tu columna de Excel cargada en Pandas
df['Texto_Limpio'] = df['Texto'].apply(limpiar_texto)

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/ciabd01/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


1. Preparación de los datos (Tokenización y Padding)

In [20]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

# Configuración básica
max_words = 1000  # Tamaño del vocabulario
max_len = 50      # Longitud máxima de cada reseña

# 1. Tokenización (Convertir palabras a números)
tokenizer = Tokenizer(num_words=max_words, lower=True)
tokenizer.fit_on_texts(df['Texto_Limpio'])
sequences = tokenizer.texts_to_sequences(df['Texto_Limpio'])

# 2. Padding (Que todas las reseñas midan lo mismo)
X = pad_sequences(sequences, maxlen=max_len)
y = df['Etiqueta'].values # 1 para positivo, 0 para negativo

1. Conversión de etiquetas de Texto a Números

In [23]:
import numpy as np

# Convertimos las palabras 'Positivo'/'Negativo' a 1 y 0
# Usamos un diccionario para el mapeo
mapeo = {'Positivo': 1, 'Negativo': 0}
df['Etiqueta_Num'] = df['Etiqueta'].map(mapeo)

# 1. Ahora sí, extraemos las etiquetas como números float32
y = np.array(df['Etiqueta_Num']).astype('float32')

# 2. Aseguramos que 'X' (secuencias) sean números
from tensorflow.keras.preprocessing.sequence import pad_sequences
X = np.array(pad_sequences(sequences, maxlen=50)).astype('float32')

print("Etiquetas preparadas correctamente (0 y 1)")

Etiquetas preparadas correctamente (0 y 1)


2. Entrenamiento del Modelo LSTM

In [24]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

# Definición del modelo
modelo = Sequential([
    Embedding(input_dim=1000, output_dim=32, input_length=50),
    LSTM(32),
    Dense(1, activation='sigmoid')
])

modelo.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Entrenamiento
modelo.fit(X, y, epochs=10, batch_size=8, validation_split=0.2)

Epoch 1/10


/home/ciabd01/anaconda3/lib/python3.13/site-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 93ms/step - accuracy: 0.7273 - loss: 0.6894 - val_accuracy: 1.0000 - val_loss: 0.6732
Epoch 2/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.8182 - loss: 0.6694 - val_accuracy: 1.0000 - val_loss: 0.6484
Epoch 3/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.8182 - loss: 0.6436 - val_accuracy: 1.0000 - val_loss: 0.6180
Epoch 4/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.8182 - loss: 0.6194 - val_accuracy: 1.0000 - val_loss: 0.5725
Epoch 5/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.8182 - loss: 0.5808 - val_accuracy: 1.0000 - val_loss: 0.5129
Epoch 6/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.8182 - loss: 0.5304 - val_accuracy: 1.0000 - val_loss: 0.4371
Epoch 7/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.8182 - loss: 0.4842 - val_accuracy: 1.0000 - val_loss: 0.3444
Epoch 8/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.8182 - loss: 0.4616 - val_accuracy: 1.0000 - val_loss: 0.2520
Epoch 9/10


3. Clasificación y Extracción de Puntos Clave (Insights)

In [ ]:
from collections import Counter

# 1. El modelo clasifica las reseñas
predicciones = (modelo.predict(X) > 0.5).astype("int32")
df['Prediccion'] = predicciones

# 2. Separar grupos
satisfechos = df[df['Prediccion'] == 1]['Texto_Limpio']
insatisfechos = df[df['Prediccion'] == 0]['Texto_Limpio']

# 3. Función para extraer los 5 puntos clave (Frecuencia)
def obtener_puntos_clave(textos):
    todas_palabras = " ".join(textos).split()
    # Filtramos palabras muy comunes o vacías si es necesario
    conteo = Counter(todas_palabras)
    return conteo.most_common(5) 

print("--- PUNTOS FUERTES (Top 5) ---")
print(obtener_puntos_clave(satisfechos))

print("\n--- PUNTOS DÉBILES (Top 5) ---")
print(obtener_puntos_clave(insatisfechos))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step
--- PUNTOS FUERTES (Top 5) ---
[('tablet', 15), ('great', 11), ('good', 7), ('easy', 6), ('books', 6)]

--- PUNTOS DÉBILES (Top 5) ---
[]


4. Lógica de Negocio: Procesar una reseña nueva

In [ ]:
def analizar_nueva_resena(texto_nuevo):
    # Limpiar y preparar
    limpio = limpiar_texto(texto_nuevo) 
    seq = tokenizer.texts_to_sequences([limpio])
    padded = pad_sequences(seq, maxlen=max_len)
    
    # Predecir
    pred = modelo.predict(padded)[0][0]
    
    if pred >= 0.5:
        return f"Positiva ({pred:.2%}) -> Añadir a Puntos Fuertes"
    else:
        return f"Negativa ({pred:.2%}) -> Añadir a Puntos Débiles"

# Prueba rápida
print(analizar_nueva_resena("The screen is amazing and very fast"))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 125ms/step
Positiva (89.73%) -> Añadir a Puntos Fuertes


Porcentaje de reseñas positivas vs negativas

In [27]:
total = len(df)
positivos = len(df[df['Etiqueta_Num'] == 1])
negativos = len(df[df['Etiqueta_Num'] == 0])

print(f"Total reseñas: {total}")
print(f"Positivas: {(positivos/total)*100:.2f}%")
print(f"Negativas: {(negativos/total)*100:.2f}%")

Total reseñas: 28
Positivas: 85.71%
Negativas: 14.29%
